In [64]:
# Standard library imports
import logging
import os
import sys
from datetime import datetime
from pprint import pprint
import json
import pandas as pd


project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, project_root)

from src.service import YouTubeAnalysisManager
from src.llm.LangChainService import HateSpeechClassification

In [65]:
yt = YouTubeAnalysisManager(llm = 'openai', model_name='gpt-5')

기존 벡터스토어 로드 완료
리트리버 초기화 완료: basic (k=5)
LLM Service Provider: openai
OpenAI LLM 서비스가 'gpt-5' 모델로 초기화되었습니다.
gpt-5


In [66]:
cursor = yt.youtube_dao.get_connection().cursor()

with open("../comment_rag_result_20250907_051348.json") as f:
    failed_comments = json.load(f)

In [67]:
cursor = yt.youtube_dao.get_connection().cursor()

cursor.execute("SELECT * FROM comments LIMIT 1")
colnames = [desc[0] for desc in cursor.description]
print("칼럼 목록:", colnames)

df = pd.DataFrame(columns=colnames)
for comment in failed_comments['failed_comments']:
    # print(comment['comment_text'])
    cursor = yt.youtube_dao.get_connection().cursor()
    
    print(comment['comment_id'])
    sql = f"SELECT * FROM comments WHERE comment_id = '{comment['comment_id']}'"
    cursor.execute(sql)
    
    result = cursor.fetchall()
    print(result)
    for row in result:
        df.loc[len(df)] = row

칼럼 목록: ['comment_id', 'video_id', 'author', 'author_channel_id', 'comment_text', 'like_count', 'published_at', 'updated_at', 'reply_count', 'is_reply', 'parent_id', 'collection_time', 'is_hate_speech', 'categories', 'similar_cases_used', 'target_group', 'hate_type', 'used_prompt']
UgxM44ARHmr2rkZGQ9d4AaABAg
[('UgxM44ARHmr2rkZGQ9d4AaABAg', 'Spq5VoREsPM', '@한은빈-u1j', 'UC2_Mbct8xp-7BIc8VbP-OXw', '처음부터다알고있었을거잖아...근데왜...국힘망해라고...어짜피빼길거이번판도빼기자는건가', 0, '2025-06-08T06:22:47Z', '2025-06-08T06:22:47Z', 0, False, '', datetime.datetime(2025, 7, 28, 11, 50, 36, 894700), False, [], [], None, None, None)]
UgxM5URyiuhNlikkuLh4AaABAg
[('UgxM5URyiuhNlikkuLh4AaABAg', '5NfpLEWWRyw', '@일이루나', 'UCjaQ8G93buSeSl7D4_Rwo6w', '사전선거=부정선거<br>사전선거 폐지!!!', 2, '2025-06-03T01:16:35Z', '2025-06-03T01:16:35Z', 0, False, '', datetime.datetime(2025, 7, 28, 11, 47, 24, 492426), False, [], [], None, None, None)]
UgxMACxFLW2vBKflSpB4AaABAg
[('UgxMACxFLW2vBKflSpB4AaABAg', 'hvAIZd8bmLo', '@박순남-d3u', 'UCB8TFDKCL_IvwbMkEeQuIZg',

In [68]:
results = []
for idx in df.index:
    print(df.loc[idx, "comment_text"])
    result = yt.comment_classifier.classify_single_comment(df.loc[idx, 'comment_text'])
    print(result)
    results.append(result)

처음부터다알고있었을거잖아...근데왜...국힘망해라고...어짜피빼길거이번판도빼기자는건가
1. (Isolated) Query Embedding: 1.1060 초
2. Full Retrieval (Embedding + Search): 0.5449 초
3. Formatting: 0.0000 초
4. Prompt Generation: 0.0016 초
content='{"prompt": null, "input_text": "\\"처음부터다알고있었을거잖아...근데왜...국힘망해라고...어짜피빼길거이번판도빼기자는건가\\"", "is_hate_speech": false, "categories": ["혐오없음"], "evidence_strength": 0.15, "reasoning": "해당 표현은 특정 정체성(성별, 연령, 인종, 종교, 출신지역, 성적지향 등)을 대상으로 한 모욕·비하·선동이 아니라 정치적 대상(국힘/정당)에 대한 부정적 표현입니다. 욕설이나 차별·폭력 선동도 포함되지 않아 정의상 혐오표현에 해당하지 않습니다.", "similar_cases_used": [], "target_group": null, "hate_type": null}' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 966, 'prompt_tokens': 1162, 'total_tokens': 2128, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 768, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-5-2025-08-07', 'system_fingerprint': None, 

KeyboardInterrupt: 

0: comment_id

1: video_id 

2: author 

3: author_channel_id 

4: comment_text

5: like_count

6: published_at

7: updated_at 

8: reply_count 

9: is_reply 

10: parent_id 

11: collection_time 

12: is_hate_speech 

13: categories 

14: similar_cases_used 

15: target_group 

16: hate_type 

17: used_prompt

In [ ]:
for _, result in enumerate(results):
    df.loc[_, 'categories'] = result.categories
    df.loc[_, 'used_prompt'] = result.prompt
    if df.loc[_, 'categories'] != ["혐오없음"]:
        print("====" * 20)

In [74]:
for idx in df.index:
    id_ = df.loc[idx, 'comment_id']
    cls = HateSpeechClassification(
            prompt=df.loc[idx, 'used_prompt'],
            input_text=df.loc[idx,'comment_text'],
            is_hate_speech=False,
            categories=df.loc[idx, 'categories'],
            similar_cases_used=[], 
            evidence_strength=0.1,
            reasoning="" 
            
    )
    
    yt.youtube_dao.update_hate_speech_analysis_by_id(
        comment_id=id_, analysis_result = cls.dict()
    )
    print("=== finish ===")

/tmp/ipykernel_381769/1622168801.py:15: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  comment_id=id_, analysis_result = cls.dict()


=== finish ===
=== finish ===
=== finish ===
=== finish ===
=== finish ===
=== finish ===
=== finish ===
=== finish ===
=== finish ===
